In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar `creditsight` sin instalarlo: el código vive en ejemplo/src
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 09 · Inferencia — del modelo registrado a una función que cualquiera puede usar

El modelo registrado vive en MLflow; la app **no consulta MLflow ni Databricks**. Lo que hace falta es:
1. **Exportar** el modelo `champion` a un archivo (`models/model.joblib`) que viaja con el repositorio.
2. **Un esquema de entrada** (`models/input_schema.json`): qué datos se pueden preguntar y qué valores son *posibles*.
3. **Una función `predict` que valida**: nunca responde en silencio con un dato imposible.
4. **La ficha del modelo**: qué es, qué rendimiento tiene (con intervalo), dónde falla y para qué **no** usarlo.

## 0. Datos y rutas

In [2]:
import json
import warnings
from datetime import datetime

import joblib
import mlflow
import numpy as np
import pandas as pd

from creditsight.config import CLASSES, COST_MATRIX, GROUP_COL, MODELS_DIR, PROCESSED_DIR, REPORTS_DIR, TARGET
from creditsight.data.cleaning import PLAUSIBLE_RANGE, clean, load_raw
from creditsight.data.split import split_by_customer
from creditsight.inference.predict import InputValidationError, predict, required_columns
from creditsight.inference.schema import build_schema
from creditsight.tracking.mlflow_utils import configure_mlflow, registered_model_name

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    donde = configure_mlflow()
NOMBRE = registered_model_name("creditsight_credit_score")
train, test = split_by_customer(clean(load_raw()), test_size=0.2)
numericas, categoricas = required_columns()
resultados = json.loads((REPORTS_DIR / "07_resultados.json").read_text(encoding="utf-8"))
decisiones = json.loads((REPORTS_DIR / "07_decisiones_congeladas.json").read_text(encoding="utf-8"))
print("MLflow:", donde, "· modelo:", NOMBRE)

MLflow: local · modelo: creditsight_credit_score


## 1. Exportar el modelo `champion`

Se pide al registro el modelo **por su alias** y se guarda en un archivo. Se exporta desde `champion` (no desde el
`final_pipeline.joblib` del notebook 07) para tener la certeza de que se entrega **lo que quedó registrado**.

In [3]:
modelo = mlflow.sklearn.load_model(f"models:/{NOMBRE}@champion")
joblib.dump(modelo, MODELS_DIR / "model.joblib", compress=3)   # sin pérdida: de ~295 MB a ~63 MB, bajo el límite de 100 MB de GitHub
tamano = (MODELS_DIR / "model.joblib").stat().st_size / 1e6
print(f"models/model.joblib guardado ({tamano:.1f} MB; GitHub rechaza archivos de más de 100 MB; por eso se guarda comprimido)")

models/model.joblib guardado (63.2 MB; GitHub rechaza archivos de más de 100 MB; por eso se guarda comprimido)


## 2. El esquema de entrada

La app y `predict` necesitan saber qué se puede preguntar: qué columnas, qué rango es **posible** y qué categorías conoce el
modelo. Dos cosas distintas:

- **Rango observado:** lo que se vio en el entrenamiento. Fuera de él, solo se **avisa**.
- **Rango posible:** lo que puede ocurrir en el mundo real (`PLAUSIBLE_RANGE`, **conocimiento del analista**, no de los
  datos: una edad de 200 es imposible aunque nunca haya aparecido). Fuera de él, la predicción **se niega**.

Se agregan tres **perfiles de ejemplo**, tomados de clientes reales del entrenamiento, para que quien abra la app pueda
empezar desde algo razonable.

In [4]:
def perfil(clase: str) -> dict:
    """Una fila real de esa clase, sin vacíos en las columnas de entrada, como valores simples."""
    candidatos = train[(train[TARGET] == clase)].dropna(subset=numericas + categoricas)
    fila = candidatos.iloc[len(candidatos) // 2]                               # una fila de en medio, no un extremo
    return {c: (round(float(fila[c]), 2) if c in numericas else str(fila[c])) for c in numericas + categoricas}

ejemplos = {"Perfil de bajo riesgo (Good)": perfil("Good"), "Perfil intermedio (Standard)": perfil("Standard"),
            "Perfil de alto riesgo (Poor)": perfil("Poor")}
esquema = build_schema(
    train, numericas, categoricas, CLASSES, cost_matrix=COST_MATRIX, valid_range=PLAUSIBLE_RANGE,
    examples=ejemplos, path=MODELS_DIR / "input_schema.json",
)
print(f"esquema guardado: {len(numericas)} numéricas, {len(categoricas)} categóricas, clases {CLASSES}")
print("rangos posibles definidos para:", ", ".join(esquema["valid_range"]))

esquema guardado: 17 numéricas, 4 categóricas, clases ['Poor', 'Standard', 'Good']
rangos posibles definidos para: Age, Annual_Income, Num_Bank_Accounts, Num_Credit_Card, Interest_Rate, Num_of_Loan, Num_of_Delayed_Payment, Num_Credit_Inquiries, Total_EMI_per_month


## 3. Probar la predicción

Tres casos que importan:
- una fila **normal** → responde;
- una fila **incompleta** (el usuario marcó "no sé" en una variable) → también responde, porque el pipeline imputa;
- una fila con un dato **imposible** → **se niega y explica por qué**.

In [5]:
normal = pd.DataFrame([ejemplos["Perfil intermedio (Standard)"]])
print("— fila normal —")
print(predict(normal, model=modelo, schema=esquema).round(3).T.to_string(header=False))

incompleta = normal.copy()
incompleta.loc[:, "Annual_Income"] = np.nan
print("\n— con un dato faltante (Annual_Income) —")
print(predict(incompleta, model=modelo, schema=esquema)[["decision", "most_likely", "needs_review"]].to_string(index=False))

imposible = normal.copy()
imposible.loc[:, "Age"] = 200
print("\n— con un dato imposible (Age = 200) —")
try:
    predict(imposible, model=modelo, schema=esquema)
    raise AssertionError("debía negarse a predecir con una edad de 200 años")
except InputValidationError as error:
    print("se niega, como debe:", error)

— fila normal —


decision        Standard
proba_Poor         0.159
proba_Standard     0.819
proba_Good         0.023
most_likely     Standard
expected_cost      0.499
needs_review       False

— con un dato faltante (Annual_Income) —


decision most_likely  needs_review
Standard    Standard         False

— con un dato imposible (Age = 200) —
se niega, como debe: Age: debe estar entre 14 y 100


## 4. Comprobar que el archivo exportado es el modelo que se evaluó

Antes de dárselo a la app: `models/model.joblib` debe dar **las mismas probabilidades** que se obtuvieron en la prueba del
notebook 07. Esto **no es una nueva lectura de la prueba para decidir algo**: es una comprobación de identidad contra las
probabilidades que ese notebook guardó cuando la abrió.

In [6]:
exportado = joblib.load(MODELS_DIR / "model.joblib")
proba_guardada, _ = joblib.load(PROCESSED_DIR / "prueba_probabilidades.joblib")
proba_ahora = exportado.predict_proba(test.drop(columns=[TARGET]))
identico = bool(np.allclose(proba_guardada, proba_ahora))
print("probabilidades de la prueba idénticas a las del notebook 07:", identico, f"(diferencia máxima {np.abs(proba_guardada - proba_ahora).max():.2e})")
assert identico, "El model.joblib exportado no es el modelo que se evaluó"

probabilidades de la prueba idénticas a las del notebook 07: True (diferencia máxima 2.22e-16)


## 5. La ficha del modelo

Una ficha dice **qué es**, **cuánto rinde (con su intervalo)** y **dónde no debe usarse**. Se genera con las cifras de
`07_resultados.json`, no a mano, para que no pueda contradecir al notebook 07.

In [7]:
iv = {r["medida"]: r for r in resultados["intervalos"]}
costo, auc = iv["costo esperado (menor es mejor)"], iv["AUC macro (mayor es mejor)"]
d_costo, d_auc = iv["costo: modelo − baseline"], iv["AUC: modelo − baseline"]
m = resultados["metricas_modelo"]
ficha = f"""# Ficha del modelo — CreditSight

**Qué es.** Clasificador de perfil crediticio (`Poor`, `Standard`, `Good`) con **{resultados["modelo_final"]}**
{"calibrado (sigmoide)" if decisiones["calibrar"] else "sin calibrar"}. Apoya la decisión de límite de crédito: **prioriza la revisión
del analista de riesgo, no decide por sí solo.** La decisión final es siempre humana.

**Cómo decide.** Calcula la probabilidad de cada clase y elige la decisión de **menor costo esperado** según la matriz de costos
(aprobar a un cliente `Poor` cuesta 10; rechazar a uno `Good`, 2), no la clase más probable.

**Datos.** `credit_score_classification.csv`: 12 500 clientes × 8 meses (100 000 filas), datos **sintéticos de ejemplo**.
Entrenado con el 80 % de los clientes; la prueba (20 %) se abrió **una sola vez** ({decisiones["fecha_y_hora"]}).
Se particiona **por cliente**, nunca por fila.

## Rendimiento en la prueba (clientes que el modelo nunca vio)

| Medida | Valor | IC 95 % (remuestrea clientes) |
|---|---|---|
| Costo esperado por cliente-mes (menor es mejor) | {costo["valor"]:.3f} | {costo["IC 95 % inferior"]:.3f} a {costo["IC 95 % superior"]:.3f} |
| AUC macro | {auc["valor"]:.3f} | {auc["IC 95 % inferior"]:.3f} a {auc["IC 95 % superior"]:.3f} |
| Costo frente al baseline (modelo − baseline) | {d_costo["valor"]:+.3f} | {d_costo["IC 95 % inferior"]:+.3f} a {d_costo["IC 95 % superior"]:+.3f} |
| AUC frente al baseline (modelo − baseline) | {d_auc["valor"]:+.3f} | {d_auc["IC 95 % inferior"]:+.3f} a {d_auc["IC 95 % superior"]:+.3f} |

Recall por clase con la regla de decisión: `Poor` {m["recall_poor"]:.1%} · `Standard` {m["recall_standard"]:.1%} · `Good` {m["recall_good"]:.1%}.
{"Supera al baseline con confianza." if resultados["supera_al_baseline_con_confianza"] else "NO supera al baseline con confianza."}
La app marca "revisar" en {resultados["revision"]["porcentaje_marcado"]:.0%} de los cliente-mes.

## Límites y usos no recomendados

- **Casi nunca decide `Good`.** La regla de costo mínimo penaliza tanto aprobar a un `Poor` que el recall de `Good` es muy bajo:
  el modelo es prudente por diseño. No sirve para encontrar a los mejores clientes.
- **Datos sintéticos de ejemplo.** No usar con clientes reales sin una validación propia con datos reales.
- **Explica al modelo, no al mundo.** Las variables que más pesan (SHAP) muestran de qué se apoya el modelo, no qué *causa*
  el comportamiento de pago.
- **Variables de uso sensible:** {", ".join(resultados["variables_sensibles"])}. Revisar con el área legal antes de producción.
- `Credit_Mix` se usa asumiendo que se conoce al momento de decidir.
- El costo depende de la matriz de costos supuesta; con otra matriz, otra decisión óptima.
"""
(MODELS_DIR / "ficha_modelo.md").write_text(ficha, encoding="utf-8")
print(ficha)

# Ficha del modelo — CreditSight

**Qué es.** Clasificador de perfil crediticio (`Poor`, `Standard`, `Good`) con **random forest**
sin calibrar. Apoya la decisión de límite de crédito: **prioriza la revisión
del analista de riesgo, no decide por sí solo.** La decisión final es siempre humana.

**Cómo decide.** Calcula la probabilidad de cada clase y elige la decisión de **menor costo esperado** según la matriz de costos
(aprobar a un cliente `Poor` cuesta 10; rechazar a uno `Good`, 2), no la clase más probable.

**Datos.** `credit_score_classification.csv`: 12 500 clientes × 8 meses (100 000 filas), datos **sintéticos de ejemplo**.
Entrenado con el 80 % de los clientes; la prueba (20 %) se abrió **una sola vez** (2026-10-28T17:25:46).
Se particiona **por cliente**, nunca por fila.

## Rendimiento en la prueba (clientes que el modelo nunca vio)

| Medida | Valor | IC 95 % (remuestrea clientes) |
|---|---|---|
| Costo esperado por cliente-mes (menor es mejor) | 0.476 | 0.456 a 0.498 |
| 

## 6. Siguiente: la app

La app carga `models/model.joblib` y `models/input_schema.json` y no consulta Databricks:

```bash
streamlit run ejemplo/app/streamlit_app.py
```

Desde la raíz del repositorio. En el formulario, `Perfil de alto riesgo (Poor)` rellena un cliente de ejemplo; marcar
*No sé* en una variable no rompe nada, y un valor imposible muestra un error claro.